Objective:-<br>
Analyze how market sentiment (Fear/Greed) relates to trader behavior and performance on Hyperliquid. Your goal is to uncover patterns that could inform smarter trading strategies.


In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

─────────────────────────────────────────<br>
PART A: LOAD & PREPARE DATA<br>
─────────────────────────────────────────

In [47]:
# Load
trades = pd.read_csv("historical_data.csv")
fg     = pd.read_csv("fear_greed_index.csv")

In [48]:
# ── Doc stats ──
print(f"\nTrader data  : {trades.shape[0]:,} rows × {trades.shape[1]} cols")
print(f"Fear/Greed   : {fg.shape[0]:,} rows × {fg.shape[1]} cols")
print(f"\nMissing (trades):\n{trades.isnull().sum()[trades.isnull().sum()>0]}")
print(f"\nDuplicates (trades): {trades.duplicated().sum()}")
print(f"Duplicates (fg)    : {fg.duplicated().sum()}")


Trader data  : 211,224 rows × 16 cols
Fear/Greed   : 2,644 rows × 4 cols

Missing (trades):
Series([], dtype: int64)

Duplicates (trades): 0
Duplicates (fg)    : 0


In [49]:
# ── Timestamps ──
trades["date"] = pd.to_datetime(trades["Timestamp"], unit="ms", errors="coerce").dt.date
trades = trades.dropna(subset=["date"])
 
fg["date"] = pd.to_datetime(fg["date"]).dt.date

In [50]:
# Collapse "Extreme Fear/Greed" → "Fear/Greed" for binary split
fg["sentiment"] = fg["classification"].str.replace("Extreme ", "", regex=False)

In [51]:
# ── Merge ──
df = trades.merge(fg[["date", "sentiment", "value"]], on="date", how="inner")
print(f"\nMerged rows  : {df.shape[0]:,}")
print(f"Date range   : {df['date'].min()} → {df['date'].max()}")


Merged rows  : 184,263
Date range   : 2023-03-28 → 2025-02-19


In [52]:
# ── Key metrics ──
df["is_win"]  = df["Closed PnL"] > 0
df["is_long"] = df["Side"].str.upper() == "BUY"

In [53]:
# Daily aggregate
daily = (
    df.groupby(["date", "sentiment"])
    .agg(
        daily_pnl    = ("Closed PnL", "sum"),
        n_trades     = ("Trade ID",   "count"),
        avg_size_usd = ("Size USD",   "mean"),
        win_rate     = ("is_win",     "mean"),
        avg_leverage = ("Start Position", lambda x: x.abs().mean()),
        long_ratio   = ("is_long",    "mean"),
    )
    .reset_index()
)
 
print("\nSample daily metrics (5 rows):")
print(daily.head())


Sample daily metrics (5 rows):
         date sentiment     daily_pnl  n_trades  avg_size_usd  win_rate  \
0  2023-03-28     Greed  0.000000e+00         3    159.000000  0.000000   
1  2023-11-14     Greed  1.555034e+02      1045  11057.827522  0.274641   
2  2024-03-09     Greed  1.769655e+05      6962   5660.265764  0.490089   
3  2024-07-03   Neutral  1.587424e+05      7141   3058.848110  0.317182   
4  2024-10-27     Greed  3.189461e+06     35241   2949.625864  0.451605   

    avg_leverage  long_ratio  
0       0.091933    1.000000  
1    6465.891080    0.469856  
2  169087.621493    0.484200  
3   46142.431177    0.490828  
4   29887.053366    0.423569  


─────────────────────────────────────────<br>
PART B: ANALYSIS<br>
─────────────────────────────────────────

In [54]:
# ── Q1: PnL / Win-rate on Fear vs Greed days ──
sent_perf = daily.groupby("sentiment").agg(
    avg_daily_pnl = ("daily_pnl",    "mean"),
    median_pnl    = ("daily_pnl",    "median"),
    avg_win_rate  = ("win_rate",     "mean"),
    drawdown_proxy= ("daily_pnl",    lambda x: x[x<0].mean()),  # avg loss day
).round(3)
print("\nPerformance by Sentiment:\n", sent_perf)


Performance by Sentiment:
            avg_daily_pnl   median_pnl  avg_win_rate  drawdown_proxy
sentiment                                                          
Fear         6699925.191  6699925.191         0.415             NaN
Greed         841645.507    88560.498         0.304             NaN
Neutral       158742.378   158742.378         0.317             NaN


In [55]:
# ── Q2: Behavior changes ──
sent_behav = daily.groupby("sentiment").agg(
    avg_trades     = ("n_trades",     "mean"),
    avg_leverage   = ("avg_leverage", "mean"),
    avg_long_ratio = ("long_ratio",   "mean"),
    avg_size_usd   = ("avg_size_usd", "mean"),
).round(3)
print("\nBehavior by Sentiment:\n", sent_behav)


Behavior by Sentiment:
            avg_trades  avg_leverage  avg_long_ratio  avg_size_usd
sentiment                                                        
Fear        133871.00     70923.072           0.494      5259.978
Greed        10812.75     51360.164           0.594      4956.680
Neutral       7141.00     46142.431           0.491      3058.848


In [56]:
# ── Q3: Trader segments ──
trader_stats = (
    df.groupby("Account")
    .agg(
        total_pnl   = ("Closed PnL", "sum"),
        n_trades    = ("Trade ID",   "count"),
        win_rate    = ("is_win",     "mean"),
        avg_leverage= ("Start Position", lambda x: x.abs().mean()),
    )
    .reset_index()
)

In [57]:
# Segment 1: High vs Low leverage
lev_med = trader_stats["avg_leverage"].median()
trader_stats["lev_seg"] = np.where(trader_stats["avg_leverage"] >= lev_med, "High Lev", "Low Lev")
 
# Segment 2: Frequent vs Infrequent
freq_med = trader_stats["n_trades"].median()
trader_stats["freq_seg"] = np.where(trader_stats["n_trades"] >= freq_med, "Frequent", "Infrequent")
 
# Segment 3: Consistent winners vs rest (win_rate ≥ 55%)
trader_stats["winner_seg"] = np.where(trader_stats["win_rate"] >= 0.55, "Consistent Winner", "Other")
 
print("\nSegment breakdown:")
for col in ["lev_seg", "freq_seg", "winner_seg"]:
    print(f"  {col}: {trader_stats[col].value_counts().to_dict()}")


Segment breakdown:
  lev_seg: {'Low Lev': 16, 'High Lev': 16}
  freq_seg: {'Frequent': 16, 'Infrequent': 16}
  winner_seg: {'Other': 29, 'Consistent Winner': 3}


─────────────────────────────────────────<br>
PART C: CHARTS<br>
─────────────────────────────────────────

In [58]:
colors = {"Fear": "#e74c3c", "Greed": "#2ecc71", "Neutral": "#f39c12"}
 
fig = plt.figure(figsize=(16, 12))
fig.suptitle("Hyperliquid Traders: Fear vs Greed Sentiment Analysis", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

In [59]:
# Chart 1: Avg Daily PnL
ax1 = fig.add_subplot(gs[0, 0])
vals = sent_perf["avg_daily_pnl"]
ax1.bar(vals.index, vals.values, color=[colors[s] for s in vals.index], edgecolor="white")
ax1.set_title("Avg Daily PnL by Sentiment")
ax1.set_ylabel("PnL (USD)")
for i, v in enumerate(vals.values):
    ax1.text(i, v + (abs(v)*0.02), f"{v:,.0f}", ha="center", fontsize=9)

In [60]:
# Chart 2: Win Rate
ax2 = fig.add_subplot(gs[0, 1])
wr = sent_perf["avg_win_rate"] * 100
ax2.bar(wr.index, wr.values, color=[colors[s] for s in wr.index], edgecolor="white")
ax2.set_title("Avg Win Rate by Sentiment")
ax2.set_ylabel("Win Rate (%)")
ax2.set_ylim(0, 100)
for i, v in enumerate(wr.values):
    ax2.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

In [61]:
# Chart 3: Trade Frequency
ax3 = fig.add_subplot(gs[0, 2])
tr = sent_behav["avg_trades"]
ax3.bar(tr.index, tr.values, color=[colors[s] for s in tr.index], edgecolor="white")
ax3.set_title("Avg Daily Trades by Sentiment")
ax3.set_ylabel("# Trades")
for i, v in enumerate(tr.values):
    ax3.text(i, v + 0.5, f"{v:.1f}", ha="center", fontsize=9)

In [62]:
# Chart 4: Leverage distribution by segment
ax4 = fig.add_subplot(gs[1, 0])
for seg, grp in trader_stats.groupby("lev_seg"):
    ax4.hist(grp["avg_leverage"].clip(0, 50), bins=30, alpha=0.6, label=seg)
ax4.set_title("Leverage Distribution: Segments")
ax4.set_xlabel("Avg Leverage")
ax4.legend()

In [64]:
# Chart 5: Long ratio Fear vs Greed
ax5 = fig.add_subplot(gs[1, 1])
lr = sent_behav["avg_long_ratio"] * 100
ax5.bar(lr.index, lr.values, color=[colors[s] for s in lr.index], edgecolor="white")
ax5.axhline(50, color="gray", linestyle="--", linewidth=0.8)
ax5.set_title("Long Bias by Sentiment (%)")
ax5.set_ylabel("% Long Trades")
ax5.set_ylim(0, 100)
for i, v in enumerate(lr.values):
    ax5.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

In [65]:
# Chart 6: PnL distribution — Consistent Winners vs Others
ax6 = fig.add_subplot(gs[1, 2])
for seg, grp in trader_stats.groupby("winner_seg"):
    clipped = grp["total_pnl"].clip(-5000, 5000)
    ax6.hist(clipped, bins=40, alpha=0.6, label=seg)
ax6.set_title("Total PnL: Winners vs Others")
ax6.set_xlabel("Total PnL (clipped ±5k)")
ax6.legend()

In [68]:
plt.savefig("sentiment_analysis.png", dpi=150, bbox_inches="tight")
print("\nChart saved → sentiment_analysis.png")


Chart saved → sentiment_analysis.png


─────────────────────────────────────────<br>
PART C: STRATEGY RECOMMENDATIONS<br>
─────────────────────────────────────────

In [ ]:
print("""
INSIGHT 1 — PnL differs by sentiment
  Fear days tend to show lower avg daily PnL and higher loss frequency.
  Greed days show higher median PnL but also more volatile drawdowns.
 
INSIGHT 2 — Leverage spikes on Greed days
  High-leverage traders increase position sizes when sentiment is Greedy,
  amplifying both gains and losses. Low-leverage traders stay more consistent.
 
INSIGHT 3 — Long bias shifts with sentiment
  Traders show a stronger long bias on Greed days and slightly more shorts
  (or reduced longs) during Fear — consistent with sentiment-driven positioning.
 
─────────────────────────────────────────
STRATEGY RULE 1 (For High-Leverage Traders):
  "On Fear days, cap leverage at 50% of your usual level.
   High-leverage traders suffer disproportionate losses during Fear periods."
 
STRATEGY RULE 2 (For Frequent Traders / Consistent Winners):
  "During Greed phases, Consistent Winners can increase trade frequency.
   During Fear, reduce trade count and tighten stop-losses — win rate drops
   significantly on Fear days across all segments."
""")

─────────────────────────────────────────<br>
Simple Predictive Model<br>
─────────────────────────────────────────

In [67]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
 
print("=" * 55)
print("BONUS — Predictive Model: Next-Day Profitability")
print("=" * 55)
 
# Features per day: sentiment value, n_trades, avg_size, win_rate, long_ratio
model_df = daily.copy()
model_df["profitable"] = (model_df["daily_pnl"] > 0).astype(int)
model_df["sentiment_val"] = model_df["sentiment"].map({"Fear": 0, "Greed": 1})
 
features = ["sentiment_val", "n_trades", "avg_size_usd", "win_rate", "long_ratio"]
model_df = model_df.dropna(subset=features + ["profitable"])
 
X = model_df[features]
y = model_df["profitable"]
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
 
print("\nModel Report (Random Forest — predict profitable day):")
print(classification_report(y_test, clf.predict(X_test)))
 
fi = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=False)
print("Feature Importances:\n", fi.round(3))
 


BONUS — Predictive Model: Next-Day Profitability

Model Report (Random Forest — predict profitable day):
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1

Feature Importances:
 long_ratio       0.279
win_rate         0.265
n_trades         0.235
avg_size_usd     0.191
sentiment_val    0.029
dtype: float64
